Content for and by IEEE Signal Processing Society. (Raul Valle & Contributors)

# Independence

> ⚠️ **Draft — pending instructor review.** The visuals below execute, but execution cannot verify proofs. An SPS instructor should vet the arguments in this notebook before it is taught. Remove this banner after review.

The final workshop of the Analysis track. Independence is the assumption hiding inside every "i.i.d." in every ML paper; Borel–Cantelli tells you which coincidences recur forever; and the laws of large numbers are the *theorems* that make sample averages — Monte Carlo, training loss, every benchmark in this curriculum — mean anything at all.

### Visual setup & helpers
*(Safe to re-run anytime.)*

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 2 — *Independence & Borel–Cantelli* (~35 min)
**Goal:** define independence for events, σ-algebras, and random variables; prove both Borel–Cantelli lemmas.
**Builds on:** [Random Variables](./Random_Variables.ipynb). &nbsp; **Feeds into:** Session 2 (laws of large numbers).

---

## Independence

💡 **Intuition.** Independence means *information about one tells you nothing about the other*: learning that $A$ happened doesn't move the odds of $B$. The multiplication rule $P(A \cap B) = P(A)P(B)$ is the algebraic fingerprint of that idea — conditioning on $A$ (dividing by $P(A)$) leaves $P(B)$ untouched. The subtlety is that for many events, independence must hold for **every sub-collection**, not just the whole batch.

**Definition.** Events $A_1, \dots, A_n$ are *independent* if for every subset of indices $\{i_1 < \cdots < i_k\}$:

$$P(A_{i_1} \cap \cdots \cap A_{i_k}) = P(A_{i_1}) \cdots P(A_{i_k}).$$

Pairwise independence is strictly weaker (classic counterexample: two fair coin flips, with $A$ = first is heads, $B$ = second is heads, $C$ = the flips agree — any two are independent, all three are not: $P(A \cap B \cap C) = \tfrac14 \ne \tfrac18$).

**Definition (the general form).** σ-algebras $\mathcal{G}_1, \dots, \mathcal{G}_n$ are independent if any choice of one event from each is independent. Random variables $X_1, \dots, X_n$ are independent if their generated σ-algebras $\sigma(X_i) = \{X_i^{-1}(B)\}$ are — equivalently, the joint law is the **product measure**: $P_{(X_1, \dots, X_n)} = P_{X_1} \otimes \cdots \otimes P_{X_n}$. Fubini's theorem then factors expectations: $E[X Y] = E[X]E[Y]$ for independent $X, Y \in L^1$ — so independent variables are uncorrelated (never the converse!).

## Borel–Cantelli

For events $A_1, A_2, \dots$, define $\{A_n \text{ i.o.}\} = \limsup_n A_n = \bigcap_{n=1}^{\infty} \bigcup_{k \ge n} A_k$ — the outcomes lying in *infinitely many* $A_n$ ("infinitely often"). Note the shape: it's the event version of $\limsup$ from [Sequences & Series](./Numerical_Sequences_and_Series.ipynb).

💡 **Intuition.** Borel–Cantelli is a zero–one dichotomy about recurring coincidences. If the probabilities $P(A_n)$ shrink fast enough to have a *finite sum*, then with probability 1 only finitely many happen — the tail of a convergent series is too small to keep producing events. If the sum *diverges* **and** the events are independent, they happen infinitely often with probability 1 — infinite total probability with no coordination cannot be dodged forever.

### Proof: Borel–Cantelli I

If $\sum_n P(A_n) < \infty$, then $P(A_n \text{ i.o.}) = 0$.

For every $n$, $\{A_k \text{ i.o.}\} \subseteq \bigcup_{k \ge n} A_k$, so by subadditivity

$$P(A_k \text{ i.o.}) \le P\Big(\bigcup_{k \ge n} A_k\Big) \le \sum_{k \ge n} P(A_k) \xrightarrow{n \to \infty} 0,$$

the tail of a convergent series. $\blacksquare$ *(No independence needed.)*

### Proof: Borel–Cantelli II

If the $A_n$ are independent and $\sum_n P(A_n) = \infty$, then $P(A_n \text{ i.o.}) = 1$.

It suffices to show $P\big(\bigcap_{k \ge n} A_k^c\big) = 0$ for each $n$ (then the union over $n$ of these null sets is null, and the complement — the i.o. event — has probability 1). Using independence of complements and $1 - x \le e^{-x}$:

$$P\Big(\bigcap_{k=n}^{m} A_k^c\Big) = \prod_{k=n}^{m} (1 - P(A_k)) \le \exp\Big(-\sum_{k=n}^{m} P(A_k)\Big) \xrightarrow{m \to \infty} 0,$$

since the exponent diverges to $-\infty$. Continuity from above finishes it. $\blacksquare$

In [2]:
# Borel–Cantelli in action: A_n = {U_n < p_n} for independent uniforms
# Σ 1/n² < ∞  → events stop;   Σ 1/n = ∞ → they never stop
N = 5000
U = rng.random(N)
nn = np.arange(1, N + 1)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 2.7), sharey=True)
for ax, p, title in [(axes[0], 1/nn**2, "$p_n = 1/n^2$ (Σ<∞): finitely many hits"),
                     (axes[1], 1/nn,    "$p_n = 1/n$ (Σ=∞): hits keep coming")]:
    hits = nn[U < p]
    ax.eventplot(hits, linelengths=0.8)
    ax.set_title(f"{title} — {len(hits)} hits in {N}")
    ax.set_xlabel("n")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1900545/334830335.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 2 — *Laws of Large Numbers* (~40 min)
**Goal:** prove the weak law with Chebyshev; state the strong law; see exactly what they license.
**Builds on:** Session 1.

---

## The Weak Law of Large Numbers

💡 **Intuition.** Why do averages settle down? Independent errors *cancel*: the variance of a sum of independent variables adds, so the variance of the **average** shrinks like $1/n$. Chebyshev converts "tiny variance" into "probably close to the mean" — and that's the whole proof. Every Monte Carlo error bar, every "$\pm$" on a benchmark, is this argument.

### Proof: WLLN (finite-variance version)

Let $X_1, X_2, \dots$ be i.i.d. with mean $\mu$ and variance $\sigma^2 < \infty$, and $\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i$. Then for every $\varepsilon > 0$, $P(|\bar{X}_n - \mu| \ge \varepsilon) \to 0$.

Linearity gives $E[\bar{X}_n] = \mu$. Independence makes variances add:

$$\mathrm{Var}(\bar{X}_n) = \frac{1}{n^2} \sum_{i=1}^{n} \mathrm{Var}(X_i) = \frac{\sigma^2}{n}.$$

Chebyshev ([Random Variables](./Random_Variables.ipynb)) then gives

$$P(|\bar{X}_n - \mu| \ge \varepsilon) \le \frac{\sigma^2}{n \varepsilon^2} \xrightarrow{n \to \infty} 0. \;\blacksquare$$

## The Strong Law

**Theorem (SLLN, Kolmogorov).** For i.i.d. $X_i$ with $E[|X_1|] < \infty$:

$$P\Big( \lim_{n \to \infty} \bar{X}_n = \mu \Big) = 1.$$

*(Proof omitted — the standard route runs through Borel–Cantelli I plus a truncation argument; see Durrett §2.4. A pleasant exercise: prove it yourself under the stronger assumption $E[X_1^4] < \infty$, using Chebyshev on fourth moments + Borel–Cantelli I.)*

The difference is real: WLLN says each *fixed large* $n$ is probably fine; SLLN says the *trajectory* $\bar{X}_n(\omega)$ converges for almost every run of the experiment — "almost surely," i.e. off a null set from [Measure Theory](./Measure_Theory.ipynb).

In [3]:
# Watch trajectories converge (SLLN): 20 independent runs of a cumulative average
n_steps, n_paths = 20_000, 20
X = rng.exponential(1.0, size=(n_paths, n_steps))          # μ = 1
avg = np.cumsum(X, axis=1) / np.arange(1, n_steps + 1)

plt.figure(figsize=(8.5, 3))
plt.plot(avg.T, linewidth=0.6, alpha=0.6)
plt.axhline(1.0, color="k", linewidth=1.2)
plt.xscale("log"); plt.ylim(0.4, 1.8)
plt.title("20 sample paths of $\\bar{X}_n$, Exponential(1): every path → μ = 1")
plt.xlabel("n (log scale)")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1900545/772346111.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [4]:
# And the WLLN rate: spread of X̄_n shrinks like 1/√n (variance like 1/n)
ns = np.array([10, 100, 1000, 10000])
trials = np.array([[rng.exponential(1.0, n).mean() for _ in range(2000)] for n in ns])

print(" n      std of X̄_n   σ/√n prediction")
for n_val, row in zip(ns, trials):
    print(f"{n_val:>6}   {row.std():.4f}       {1/np.sqrt(n_val):.4f}")

 n      std of X̄_n   σ/√n prediction
    10   0.3109       0.3162
   100   0.0996       0.1000
  1000   0.0319       0.0316
 10000   0.0100       0.0100


### What independence buys — and what breaks without it

The variance-addition step $\mathrm{Var}(\sum X_i) = \sum \mathrm{Var}(X_i)$ *needs* uncorrelatedness. With correlated samples the $1/n$ shrinkage stalls:

In [5]:
# AR(1)-correlated "samples" (ρ = 0.95) vs independent ones: same marginal variance
n = 5000
rho = 0.95
eps = rng.standard_normal((200, n))
ar = np.zeros_like(eps)
for t in range(1, n):
    ar[:, t] = rho * ar[:, t-1] + np.sqrt(1 - rho**2) * eps[:, t]

for name, data in [("independent", eps), ("correlated (ρ=0.95)", ar)]:
    means = data.mean(axis=1)
    print(f"{name:22s} std of X̄_5000 across 200 runs: {means.std():.4f}")
print("→ correlated samples are worth far fewer 'effective' samples —")
print("  the reason time-series folks and MCMC users obsess over effective sample size.")

independent            std of X̄_5000 across 200 runs: 0.0134
correlated (ρ=0.95)    std of X̄_5000 across 200 runs: 0.0835
→ correlated samples are worth far fewer 'effective' samples —
  the reason time-series folks and MCMC users obsess over effective sample size.


## The Analysis Track, Closed

The arc, start to finish: [ℝ exists](./Real_Number_Systems.ipynb) → [its topology](./Basic_Topology.ipynb) → [limits work](./Numerical_Sequences_and_Series.ipynb) → [sets can be measured](./Measure_Theory.ipynb) → [probability is a measure](./Random_Variables.ipynb) → **averages converge** (this notebook). Every empirical mean in this curriculum now stands on proved ground.

---
## Where next

- [Machine Learning](../../Intro_Mach_Learn/README.md) — training is minimizing an *empirical* average that LLN ties to the true risk.
- [Time Series](../../Intro_Time_Series/README.md) — what changes when independence fails by design.
- [Adaptive Filtering: Kalman](../../Intro_Time_Series/Intro_AdFilt_KF.ipynb) — optimal estimation when you know the correlations.